# Kaggle Test Evaluation

Notebook nay dung de evaluate cac checkpoint da train tren tap `test-00000-of-00001.parquet` ngay tren Kaggle.

Neu chay sau all-in-one notebook trong cung session, no doc truc tiep `/kaggle/working/report_experiment_outputs`. Neu chay session moi, attach/upload `allinone_report_results.zip`, notebook se tu unzip.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
RUNS_ROOT = WORKING / 'test_eval_runs'
KERNEL_OUTPUT_DIR = WORKING / 'kernel_outputs'
KAGGLE_KERNEL_OUTPUTS = [
    'anhnguyen0812/nlp-finetune',
    'anhnguyen0812/nlp-sumarization-causal-lm',
]
DOWNLOAD_KERNEL_OUTPUTS = True
OUT_DIR = WORKING / 'test_eval_outputs'
TEST_BASENAME = 'test-00000-of-00001.parquet'
MAX_TEST_SAMPLES = None  # dat 50 de smoke test nhanh

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

WORKING.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


In [ ]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)
run('nvidia-smi', check=False)


## Prepare Runs And Test File

Notebook se tai output tu `KAGGLE_KERNEL_OUTPUTS`, unzip cac file ket qua, gom moi run co `resolved_config.json` va `best/*.safetensors` vao `/kaggle/working/test_eval_runs`, roi tim `test-00000-of-00001.parquet`.


In [ ]:
def find_files(root, name):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(name))

def download_kernel_outputs():
    if not DOWNLOAD_KERNEL_OUTPUTS:
        return
    KERNEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for kernel in KAGGLE_KERNEL_OUTPUTS:
        dest = KERNEL_OUTPUT_DIR / kernel.split('/')[-1]
        dest.mkdir(parents=True, exist_ok=True)
        if any(dest.iterdir()):
            print('KERNEL OUTPUT EXISTS:', kernel, dest)
            continue
        code = run(f'kaggle kernels output {kernel} -p {dest}', cwd=WORKING, check=False)
        if code != 0:
            print('WARN: could not download kernel output:', kernel)

def unzip_result_zips():
    zip_candidates = []
    for root in [KERNEL_OUTPUT_DIR, Path('/kaggle/input'), Path('/kaggle/working')]:
        if root.exists():
            zip_candidates.extend(p for p in root.rglob('*.zip') if p.name != 'test_eval_results.zip')
    zip_candidates = sorted(set(zip_candidates))
    if not zip_candidates:
        print('No result zip found. Will try to use existing run folders.')
        return
    for zip_path in zip_candidates:
        print('UNZIP', zip_path, '->', WORKING)
        with ZipFile(zip_path) as zf:
            zf.extractall(WORKING)

def has_model_artifacts(run_dir):
    best = run_dir / 'best'
    if not best.exists():
        return False
    return any((best / name).exists() for name in ['adapter_model.safetensors', 'model.safetensors'])

def collect_run_dirs():
    search_roots = [WORKING]
    if Path('/kaggle/input').exists():
        search_roots.append(Path('/kaggle/input'))
    found = []
    for root in search_roots:
        for config_path in root.rglob('resolved_config.json'):
            run_dir = config_path.parent
            if repo in run_dir.parents or OUT_DIR in run_dir.parents or RUNS_ROOT in run_dir.parents:
                continue
            if has_model_artifacts(run_dir):
                found.append(run_dir)
    unique = []
    seen = set()
    for run_dir in sorted(found):
        key = str(run_dir.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(run_dir)
    return unique

def build_merged_runs_root(run_dirs):
    if RUNS_ROOT.exists():
        shutil.rmtree(RUNS_ROOT)
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, run_dir in enumerate(run_dirs):
        name = run_dir.name
        dest = RUNS_ROOT / name
        if dest.exists():
            dest = RUNS_ROOT / f'{name}_{idx}'
        try:
            os.symlink(run_dir, dest, target_is_directory=True)
        except OSError:
            shutil.copytree(run_dir, dest)

def find_test_file():
    candidates = find_files('/kaggle/input', TEST_BASENAME) + find_files('/kaggle/working', TEST_BASENAME)
    if not candidates:
        raise FileNotFoundError(f'Khong thay {TEST_BASENAME}. Hay attach/upload Kaggle dataset co test parquet.')
    return candidates[0]

download_kernel_outputs()
unzip_result_zips()
run_dirs = collect_run_dirs()
if not run_dirs:
    raise FileNotFoundError('Khong thay run folder nao co resolved_config.json va best/*.safetensors. Hay kiem tra output train da export model adapter/full model chua.')
build_merged_runs_root(run_dirs)
TEST_FILE = find_test_file()
merged_run_dirs = sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir() and (p / 'resolved_config.json').exists())
print('TEST_FILE:', TEST_FILE)
print('RUNS_ROOT:', RUNS_ROOT)
print('RUNS:', [p.name for p in merged_run_dirs])


## Evaluate On Test

Mac dinh chay full test. Neu muon smoke test, sua `MAX_TEST_SAMPLES = 50` o cell dau.


In [ ]:
args = f'--runs_root {RUNS_ROOT} --test_file {TEST_FILE} --out_dir {OUT_DIR}'
if MAX_TEST_SAMPLES:
    args += f' --max_test_samples {MAX_TEST_SAMPLES}'
run(f'{sys.executable} -u -m vn_summarization.evaluate_runs_on_test {args}', cwd=repo)


In [ ]:
zip_path = WORKING / 'test_eval_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUT_DIR.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
print('TEST CSV', OUT_DIR / 'test_results.csv')
print('TEST MD', OUT_DIR / 'test_results.md')
for file in sorted(files):
    print(file)
